In [1]:
import pandas as pd
import numpy as np 
import seaborn as sns
import joblib
from IPython.display import display
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, timedelta

# df = pd.read_csv(r"C:\Users\DeLL\Desktop\final Year project\Hybrid-Ai-Assistance-Page-Replacement-System\ai_dev\Software_integration\linux_and_ai_design\AI_module\model\data\swap_log_1.csv")
df = pd.read_csv("../data/swap_log_1.csv")
df.head(5)
df.dropna(inplace=True)

#------------------ data formatting ---------------------
def hex_to_int(hex_str):
    """Convert hex string to integer efficiently - handles multiple formats"""
    return hex_str.apply(lambda x: int(str(x).replace("0x",''),16)).astype(np.int64)

# Apply conversions more efficiently with error handling
hex_columns = ['PID', 'VA', 'PFN', 'mapping']
for col in hex_columns:
    df[col] = hex_to_int(df[col])

# Convert numeric columns with error handling
df['start_ns'] = pd.to_numeric(df['start_ns'], errors='coerce').fillna(0).astype(np.int64)
df['latency_ns'] = pd.to_numeric(df['latency_ns'], errors='coerce').fillna(0).astype(np.int64)

# Remove any rows that might have become NaN during conversion
df = df.dropna()

# #make int_to_hex
df['PFN_Hex'] = df['PFN'].apply(lambda x: f'{ x & 0xFFFFFFFFF :x}')
df['VA_Hex'] = df['VA'].apply(lambda x: f'{ x & 0xFFFFFFFFF :x}')
df['mapping_Hex'] = df['mapping'].apply(lambda x: f'{ x & 0xFFFFFFFFF :x}')

df["timestamp"] = pd.to_datetime(df["start_ns"]).dt.strftime("%H:%M:%S")
df = df.sort_values(by="start_ns", ascending=True).reset_index(drop=True)


In [1]:
# time = 60
# df['time_bin'] = df['start_ns'].apply(lambda x: (x/1e9)//time).astype(np.int64)
# df['time_frame'] = pd.to_datetime(df["time_bin"]*1e9*time).dt.strftime("%H:%M:%S")

# #short By START-NS in TIME
# df = df.sort_values(by="timestamp", ascending=True).reset_index(drop=True)

# fig1 = px.scatter_3d(
#     df,
#     x="timestamp",
#     y="PFN",
#     z="folio_index",
#     # color=df["PID"].astype(str),
#     hover_data=["COMM", "VA", "PFN", "folio_index"],
#     title="PFN Distribution (Group by PID)",
#     template="plotly_dark",
#     opacity=0.7,
#     animation_frame="time_frame",   # ⬅ Animation by minute
#     animation_group='start_ns'
# )

# # Make visualization clearer
# fig1.update_traces(marker=dict(size=3, opacity=0.65))

# # Fix camera + remove gridlines + set axis titles
# fig1.update_layout(
#     scene_camera=dict(eye=dict(x=1.7, y=1.7, z=0.7)),
#     scene=dict(
#         xaxis=dict(title="Start Time (sec)", showgrid=True),
#         yaxis=dict(title="Page Frame Number (VA)", showgrid=True),
#         zaxis=dict(title="Mapping (Hex)", showgrid=True),
#     ),
#     margin=dict(l=0, r=0, b=0, t=40)
# )
# fig1.show()


# fig2 = px.scatter_3d(
#     df,
#     x="timestamp",
#     y="VA_Hex",
#     z="mapping_Hex",
#     # color=df["PID"].astype(str),
#     hover_data=["COMM", "VA", "PFN", "folio_index"],
#     title="VA Distribution (Group by PID)",
#     template="plotly_dark",
#     opacity=0.7,
#     animation_frame="time_frame",   # ⬅ Animation by minute
#     animation_group="time_frame"                  # ⬅ Smooth movement per PID
# )

# # Make visualization clearer
# fig2.update_traces(marker=dict(size=3, opacity=0.65))

# # Fix camera + remove gridlines + set axis titles
# fig2.update_layout(
#     scene_camera=dict(eye=dict(x=1.7, y=1.7, z=0.7)),
#     scene=dict(
#         xaxis=dict(title="Start Time (sec)", showgrid=True),
#         yaxis=dict(title="Virtiual Address (VA)", showgrid=True),
#         zaxis=dict(title="Mapping (Hex)", showgrid=True),
#     ),
#     margin=dict(l=2, r=2, b=2, t=40)
# )
# fig2.show()

# fig3 = px.scatter(df, x="timestamp", y="folio_index",
#                   hover_data=["PID","COMM", "VA_Hex", "PFN_Hex", "folio_index","mapping_Hex"],
#                   title="Folio_index Distribution (Group by PID)",
#                   template="plotly_dark")
# fig3.update_layout(xaxis_title="start_sec (time)", yaxis_title="Folio_index")
# fig3.show()

# fig4 = px.scatter(df, x="timestamp", y="mapping_Hex",
#                   hover_data=["PID","COMM", "VA_Hex", "PFN_Hex", "folio_index","mapping_Hex"],
#                   title="maping Distribution (Group by PID)",
#                   template="plotly_dark")
# fig4.update_layout(xaxis_title="start_sec (time)", yaxis_title="Folio_index")
# fig4.show()

# fig5 = px.scatter(df, x="start_ns", y="PFN",
#                   hover_data=["PID","COMM", "VA_Hex", "PFN_Hex", "folio_index","mapping_Hex"],
#                   title="PFN Distribution (Group by PID)",
#                 #   color='mapping',
#                 #   animation_frame='PID',
#                   template="plotly_dark")
# fig5.update_layout(xaxis_title="start_sec (time)", yaxis_title="PFN")
# fig5.show()


### Visualization of VA into split L1, L2, L3, L4

    L1 = va.apply(lambda x: f'{(x >> 12)& 0x1FF: X}').unique()
    L2 = va.apply(lambda x: f'{(x >> 21)& 0x1FF: X}').unique() 
    L3 = va.apply(lambda x: f'{(x >> 30)& 0x1FF: X}').unique() 
    L4 = va.apply(lambda x: f'{(x >> 39)& 0x1FF: X}').unique()

In [ ]:
# TL1=[]
# TL2=[]
# TL3=[]
# TL4=[]
# TP = 0
# for pid in df['PID'].unique():
#     va = df[df['PID']==pid]['VA']
#     comm = df[df['PID']==pid]['COMM'].unique()
#     if len(va) > 0:
#         # 1) Page-table indices निकालें (x86-64, 4-level)
#         L1 = va.apply(lambda x: f'{(x >> 12)& 0x1FF: X}').unique()
#         L2 = va.apply(lambda x: f'{(x >> 21)& 0x1FF: X}').unique() 
#         L3 = va.apply(lambda x: f'{(x >> 30)& 0x1FF: X}').unique() 
#         L4 = va.apply(lambda x: f'{(x >> 39)& 0x1FF: X}').unique() 
#         mapping = df[df['PID']==pid]['mapping'].apply(lambda x: f'{x & 0xFFFFFFFF:x}').unique()

#         for val in L1:
#             if val not in TL1:
#                 TL1.append(val)
#         for val in L2:
#             if val not in TL2:
#                 TL2.append(val)
#         for val in L3:
#             if val not in TL3:
#                 TL3.append(val)
#         for val in L4:
#             if val not in TL4:
#                 TL4.append(val)
#         TP += 1

#         print(f"PID = '{pid}' | process name ='{comm[0]}' | TOTAL SIZE:'{len(va)}' | {len(L1),len(L2),len(L3),len(L4)}")
#         print(L1,'\n',L2,'\n',L3,'\n',L4,'\n',mapping)
        


# print(f"toatal process = {TP} |total L1 = {len(TL1)} | total L2 = {len(TL2)} |total L3 = {len(TL3)} | total L4 = {len(TL4)}")
# print(TL1,'\n',TL2,'\n',TL3,'\n',TL4)

In [2]:
# va descirbe in different sub regional parts as l1, l2, l3, l4

df['Va_L1'] = df['VA'].apply(lambda x: f'{(x >> 12)& 0x1FF}')
df['Va_L2'] = df['VA'].apply(lambda x: f'{(x >> 21)& 0x1FF}') 
df['Va_L3'] = df['VA'].apply(lambda x: f'{(x >> 30)& 0x1FF}') 
df['Va_L4'] = df['VA'].apply(lambda x: f'{(x >> 39)& 0xFFFFFFFF}') 

# page frame number are descride to predict actaul pfn using classified into sub region due to avoid pfn randomness. 

df['PFN_Top_region']=df['PFN'].apply(lambda x:(x >> 20) & 0xF)
df['PFN_Top_region_4']=df['PFN'].apply(lambda x:(x >> 12) & 0x1FF)
df['PFN_slice_4']=df['PFN'].apply(lambda x:(x >> 16) & 0xF )
df['PFN_slice_3']=df['PFN'].apply(lambda x:(x >> 12) & 0xF )
df['PFN_slice_2']=df['PFN'].apply(lambda x:(x >> 8) & 0xF )
df['PFN_slice_1']=df['PFN'].apply(lambda x:(x >> 4) & 0xF )
df['PFN_slice_0']=df['PFN'].apply(lambda x: x  & 0xF )


<h4 style='text-transform:uppercase; font-weight:700; color:#743474aa;'>write a code to convert pfn sub classification into hex value.</h4> 

In [3]:
sample = df[df['latency_ns']>1e6].copy()

sample['PFN_Top_region']=sample['PFN'].apply(lambda x:f'{(x >> 20) & 0xFF}')
sample['PFN_slice_4']=sample['PFN'].apply(lambda x:f'{(x >> 16) & 0xF :x}')
sample['PFN_slice_3']=sample['PFN'].apply(lambda x:f'{(x >> 12) & 0xF :x}')
sample['PFN_slice_2']=sample['PFN'].apply(lambda x:f'{(x >> 8) & 0xF :x}')
sample['PFN_slice_1']=sample['PFN'].apply(lambda x:f'{(x >> 4) & 0xF :x}')
sample['PFN_slice_0']=sample['PFN'].apply(lambda x: f'{x  & 0xF :x}')
(
sample.shape,
sample['PFN_slice_0'].unique(),
sample['PFN_slice_1'].unique(),
sample['PFN_slice_2'].unique(),
sample['PFN_slice_3'].unique(),
sample['PFN_slice_4'].unique(),
sample['PFN_Top_region'].unique(),
)

((1100, 23),
 array(['1', '2', '5', 'd', '7', 'c', 'b', '4', '8', '9', '0', 'f', 'e',
        'a', '6', '3'], dtype=object),
 array(['f', '5', 'd', '3', '6', '0', 'b', '2', 'e', '4', '1', '9', '8',
        'a', '7', 'c'], dtype=object),
 array(['2', 'e', '7', '8', '1', 'f', 'a', 'c', 'd', '3', 'b', '6', '0',
        '4', '9', '5'], dtype=object),
 array(['e', '4', '9', '5', '3', '7', '2', '1', 'd', '6', 'a', '0', 'f',
        'b', 'c', '8'], dtype=object),
 array(['9', '1', '5', '3', 'd', 'b', '0', 'a', '4', '8', 'c', '6', '2',
        '7'], dtype=object),
 array(['0', '1'], dtype=object))

In [4]:
fig5 = px.line(df, x="start_ns", y="PFN_slice_0",
                  hover_data=["PID","COMM", "VA_Hex", "PFN_Hex", "folio_index","mapping_Hex"],
                  title="PFN_slice_1 Distribution (Group by PID)",
                  # color=df['PID'].astype('str'),
                #   animation_frame=df['PFN_slice_4'].astype('str'),
                  template="plotly_dark")
fig5.update_layout(xaxis_title="start_sec (time)", yaxis_title="PFN")
fig5.show()

### Write a function to display mapping relation with respect to groupKey

In [5]:
def display_mapping_relation(data, mapping_label, group_key):
    """
    Display mapping relations grouped by given columns.
    """

    if not isinstance(group_key, list):
        raise ValueError("group_key must be a list of column names.")

    mapping_values = data[mapping_label].unique()

    for val in mapping_values:
        sample = (
            data[data[mapping_label] == val]
            .groupby(group_key)[[mapping_label]]
            .count()
            .rename(columns={mapping_label: "count"})
            .sort_values("count", ascending=False)
            .reset_index()
        )

        print(f"\n{'-'*130}")
        print(f"  View {mapping_label} = {val}  \n  grouped by: {group_key}")
        print(f"{'-'*130}\n")
        print(sample)



In [7]:

Group_Key = ['PID','COMM']
mapping_label = 'Va_L3'
# display_mapping_relation(df,group_key=Group_Key, mapping_label= mapping_label)

### Filter mapping using threshold of min(20) with w.r.t GroupKey

In [14]:
# Write a Function to Fitering mapping relation base on threshold. 

def filter_mapping_with_threshold(data=pd.DataFrame,mapping_label = str, groupKey=list, threshold = 20):
    counts = (
    data.groupby(groupKey)[[mapping_label]]
    .count()
    .rename(columns={mapping_label:'count'})
    .reset_index()
    )

    valid_group = counts[counts['count']>threshold]
    data = data.merge(valid_group, on=Group_Key, how='inner').drop('count', axis=1)

    return data,valid_group

filter_df , group = filter_mapping_with_threshold(df,mapping_label='mapping',groupKey=Group_Key,threshold=40)
# display_mapping_relation(filter_df, group_key=Group_Key, mapping_label= mapping_label)

In [15]:
filter_df.columns

Index(['PID', 'COMM', 'VA', 'PFN', 'mapping', 'folio_index', 'start_ns',
       'latency_ns', 'PFN_Hex', 'VA_Hex', 'mapping_Hex', 'timestamp', 'Va_L1',
       'Va_L2', 'Va_L3', 'Va_L4', 'PFN_Top_region', 'PFN_Top_region_4',
       'PFN_slice_4', 'PFN_slice_3', 'PFN_slice_2', 'PFN_slice_1',
       'PFN_slice_0'],
      dtype='object')

In [17]:
display(group.reset_index(drop=True))
print(f'shape of filter data : {filter_df.shape} ')

,PID,COMM,count
0,5376,upowerd,86
1,5926,pipewire-pulse,200
2,5970,pipewire-pulse,400
3,6240,pipewire-pulse,93
4,8593,gnome-shell,200
5,8773,gnome-shell,49
6,8817,gsd-housekeepin,175
7,9012,gsd-housekeepin,70
8,10631,firefox,200
9,12326,firefox,200


shape of filter data : (5786, 23) 


In [18]:
gk = ['PID','COMM','Va_L4','Va_L3','Va_L2','mapping_Hex','folio_index']
filter_df['PFN_delta'] = filter_df.groupby(gk)[['PFN']].diff(1).fillna(0).astype('int64')
filter_df['PFN_delta'].unique().shape

(1516,)

In [18]:
idx = 3
group = filter_df.groupby('PID')
pid = filter_df['PID'].unique()
print(group.get_group(pid[idx]).shape)
pid_df = group.get_group(pid[idx])
mapping = pid_df['mapping'].unique()
for mapping in pid_df['mapping'].unique():
    sample = pid_df[pid_df['mapping']==mapping]
    for index in sample['folio_index'].unique():
        sample_df = sample[sample['folio_index']==index]
        if sample_df.shape[0] >9:
            print(f'pid : {pid[idx]} | mapping : { mapping } | index : {index} | start =>')
            print(sample_df[['PID','Va_L3','Va_L2','mapping_Hex','Va_L1','folio_index','PFN','PFN_delta']].reset_index(drop=True))
            fig5 = px.line(sample_df, x="start_ns", y="PFN_delta",
                            hover_data=["PID", "VA_Hex", "PFN_Hex", "folio_index","mapping_Hex"],
                            title="PFN_slice_1 Distribution (Group by PID)",
                            color=sample_df['PID'].astype('str'),
                            # animation_frame=df['PFN_slice_4'].astype('str'),
                            template="plotly_dark")
            fig5.update_layout(xaxis_title="start_sec (time)", yaxis_title="PFN")
            fig5.show()

(36, 26)
pid : 8809 | mapping : 3741133965 | index : 10.0 | start =>
     PID Va_L3 Va_L2 mapping_Hex Va_L1  folio_index      PFN  PFN_delta
0   8809    16   498    defd2c8d   293         10.0  1120245          0
1   8809    16   498    defd2c8d   293         10.0  1172507      52262
2   8809    16   498    defd2c8d   293         10.0   797352    -375155
3   8809    16   498    defd2c8d   293         10.0  1050470     253118
4   8809    16   498    defd2c8d   293         10.0   345298    -705172
5   8809    16   498    defd2c8d   293         10.0  1226266     880968
6   8809    16   498    defd2c8d   293         10.0  1244432      18166
7   8809    16   498    defd2c8d   293         10.0   822552    -421880
8   8809    16   498    defd2c8d   293         10.0  1104247     281695
9   8809    16   498    defd2c8d   293         10.0  1237368     133121
10  8809    16   498    defd2c8d   293         10.0   196766   -1040602
11  8809    16   498    defd2c8d   293         10.0  1129014     93

In [19]:
filter_df['lat_cluster'] = pd.cut(
    filter_df['latency_ns'],
    bins=[0, 1e5, 2*1e5, 2*1e6, 15*1e6, 1e9],
    labels=[0,1,2,3,4]
)
filter_df['lat_cluster'].value_counts()


lat_cluster
0    2773
2    1947
1     558
3     401
4     107
Name: count, dtype: int64

In [24]:
sample_set = filter_df[(filter_df['latency_ns']>1e3)&(filter_df['latency_ns']<1e9*1.5)]
print(sample_set.shape)
fig5 = px.scatter(sample_set, x="start_ns", y="latency_ns",
                hover_data=["PID", "VA_Hex", "PFN_Hex", "folio_index","mapping_Hex"],
                title="latency (Group by PID)",
                color=sample_set['lat_cluster'].astype('str'),
                # animation_frame=df['PFN_slice_4'].astype('str'),
                template="plotly_dark")
fig5.update_layout(xaxis_title="PFN", yaxis_title="latency_ms")
fig5.show()


(5563, 25)


In [ ]:
# pid = filter_df['PID'].unique()
# for pid_id in pid:
#     sample_data = filter_df[filter_df['PID']==pid_id]
#     mapping = sample_data['mapping'].unique()
#     print(f'\nmapping count= {mapping.shape[0]} | unique-mapping = {mapping}')
#     Vaild_sample = []
#     for mapping_id in mapping:
#         min_pfn = sample_data['PFN'].min()
#         sample = sample_data[sample_data['mapping']==mapping_id].groupby(['PID','COMM','mapping','Va_L4','Va_L3','Va_L2','start_ns','Va_L1','folio_index','PFN_Hex','PFN_delta','PFN_Top_region','PFN_slice_4','PFN_slice_3','PFN_slice_2','PFN_slice_1','PFN_slice_0'])[['PFN']].count().rename(columns={'PFN':"count"})
#         print("\n")
#         # print(sample)

In [25]:
filter_df

,PID,COMM,VA,PFN,mapping,folio_index,start_ns,latency_ns,PFN_Hex,VA_Hex,...,Va_L4,PFN_Top_region,PFN_Top_region_4,PFN_slice_4,PFN_slice_3,PFN_slice_2,PFN_slice_1,PFN_slice_0,PFN_delta,lat_cluster
0,5970,pipewire-pulse,123325598093312,647921,534721225,0.0,0,1121761,9e2f1,9fa775000,...,224,0,158,9,14,2,15,1,0,2
1,5970,pipewire-pulse,123325598097408,85586,534721225,1.0,14168,1135929,14e52,9fa776000,...,224,0,20,1,4,14,5,2,0,2
2,5970,pipewire-pulse,123325598101504,366549,534721225,2.0,18175,1139936,597d5,9fa777000,...,224,0,89,5,9,7,13,5,0,2
3,5970,pipewire-pulse,123325598105600,219197,534721225,3.0,21098,1142859,3583d,9fa778000,...,224,0,53,3,5,8,3,13,0,2
4,5970,pipewire-pulse,123325598109696,85607,534721225,4.0,24077,1145838,14e67,9fa779000,...,224,0,20,1,4,14,6,7,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5781,92552,Renderer,128369847750656,669925,2039978266,874.0,38696584860659,1486,a38e5,6f2cd000,...,233,0,163,10,3,8,14,5,0,0
5782,92552,Renderer,128369847754752,561174,2039978266,875.0,38696584866640,1486,89016,6f2ce000,...,233,0,137,8,9,0,1,6,0,0
5783,92552,Renderer,128369847758848,561586,2039978266,876.0,38696584872332,1486,891b2,6f2cf000,...,233,0,137,8,9,1,11,2,0,0
5784,92552,Renderer,128369847767040,443150,2039978266,878.0,38696584883293,1486,6c30e,6f2d1000,...,233,0,108,6,12,3,0,14,0,0


In [26]:
filter_df.to_csv('filter_Swap_log.csv', index=False)